# 选修E10 · Day 1 上机：Agent经济基础--Agent作为经济主体

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **mesa** 构建Agent经济仿真--买方Agent/卖方Agent两类经济主体通过A2A协商交易，涌现市场价格/财富分布/存活率
2. 用 **networkx** 分析Agent交易网络拓扑--密度/聚类系数/PageRank经济影响力
3. 用 **numpy-financial** 计算Agent-as-Worker的NPV/IRR，量化Agent作为经济主体的投资价值
4. 理解Agent经济三层模型和贝叶斯Agent决策--Agent用贝叶斯更新估计公平价格
5. 建立天道推演×多Agent仿真的同构认知--仿真本质是计算化的天道推演沙盘

## 真实库与真实数据
- **mesa**（agent-based modeling 框架）：构建Agent经济仿真
- **networkx**（图网络分析）：Agent交易网络拓扑
- **numpy-financial**（金融计算）：Agent经济价值NPV/IRR
- **pandas + matplotlib**：仿真结果分析与可视化
- **真实经济参数**：A2A协议费10%、Token定价$5/1M（GPT-4o真实定价）、推理成本约束

> 详见 data/README.md

## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 所有库（mesa/networkx/numpy-financial/pandas/matplotlib/numpy）均为本地可用库，不需要API Key。

In [ ]:
# !pip install mesa networkx numpy-financial pandas matplotlib numpy -q

import warnings
warnings.filterwarnings('ignore')

import mesa
import numpy as np
import pandas as pd
import networkx as nx
import numpy_financial as npf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mesa.datacollection import DataCollector

print(f"mesa {mesa.__version__} | networkx {nx.__version__}")
print("Agent经济仿真环境就绪")

## 1. 真实经济参数

Agent经济仿真的参数基于真实世界的经济数据：

| 参数 | 值 | 真实来源 |
|------|-----|---------|
| A2A协议费率 | 10% | Agent间去中心化交易协议费率 |
| Token定价 | $5/1M tokens | GPT-4o input 定价（OpenAI 2024-2025定价页） |
| 每次A2A协商推理token | 500 tokens | Agent协商/比价/决策的合理token消耗 |
| 推理成本/协商 | ~$0.0025 | 500 tokens × $5/1M |

**推理成本是Agent经济的核心约束**--AI Agent每次A2A协商都消耗token。

In [ ]:
# 真实经济参数（可追溯来源）
# A2A协议费率 10%: 去中心化Agent交易协议费率
A2A_PROTOCOL_FEE_RATE = 0.10
# Token定价: GPT-4o input ~$5/1M tokens (OpenAI 2024-2025定价页)
TOKEN_PRICE_PER_1M = 5.0
# 每次A2A协商推理token消耗（Agent协商/比价/决策）
TOKENS_PER_NEGOTIATION = 500
# 每次A2A协商的推理成本
REASONING_COST_PER_NEGOTIATION = (TOKENS_PER_NEGOTIATION / 1_000_000) * TOKEN_PRICE_PER_1M

print(f"A2A协议费率: {A2A_PROTOCOL_FEE_RATE*100:.0f}%")
print(f"推理成本/协商: ${REASONING_COST_PER_NEGOTIATION:.4f} ({TOKENS_PER_NEGOTIATION} tokens x ${TOKEN_PRICE_PER_1M}/1M)")
print(f"  -> 这是Agent每次A2A协商的净利润约束")

## 2. TODO 1：买方Agent（贝叶斯价格估计）

**买方Agent** 是Agent经济中的需求方（营销映射：品牌Agent买广告位）。

核心属性：
- `wealth`：预算（初始1000）
- `price_mu` / `price_var`：贝叶斯价格信念（Normal先验）
- `n_obs`：观测次数

**行为逻辑**：
1. 70%概率选择最低价卖方（exploit），30%概率随机选择（explore）
2. 只接受低于"后验均值+1标准差"的价格（贝叶斯决策）
3. 用观测价格更新贝叶斯后验（共轭正态更新）
4. 预算耗尽则破产

**贝叶斯更新**：Agent作为经济主体在不确定市场中学习公平价格。

In [ ]:
# 买方Agent：贝叶斯价格估计 + A2A协商购买
class BuyerAgent(mesa.Agent):
    """买方Agent（营销映射：品牌Agent买广告位）。
    使用贝叶斯共轭正态更新估计公平市场价格。"""
    def __init__(self, model, initial_budget=1000.0, demand=1):
        super().__init__(model)
        self.wealth = initial_budget
        self.demand = demand
        self.alive = True
        self.purchases = 0
        self.total_spent = 0.0
        # 贝叶斯先验：公平价格 ~ N(mu=20, var=25)
        self.price_mu = 20.0
        self.price_var = 25.0
        self.n_obs = 0
        self.observation_var = 10.0  # 观测噪声方差
    def update_price_belief(self, observed_price):
        """共轭正态后验更新（已知观测方差）。"""
        prior_precision = 1.0 / self.price_var
        obs_precision = 1.0 / self.observation_var
        posterior_precision = prior_precision + obs_precision
        self.price_mu = (prior_precision * self.price_mu + obs_precision * observed_price) / posterior_precision
        self.price_var = 1.0 / posterior_precision
        self.n_obs += 1
    def step(self):
        if not self.alive:
            return
        sellers = [a for a in self.model.agents
                   if isinstance(a, SellerAgent) and a.alive and a.supply > 0]
        if not sellers:
            return
        # 有限理性：70%exploit（最低价）/ 30%explore（随机）
        if self.random.random() < 0.30:
            chosen = self.random.choice(sellers)
        else:
            chosen = min(sellers, key=lambda s: s.price)
        # 贝叶斯决策：只接受低于"后验均值+1标准差"的价格
        fair_price_upper = self.price_mu + np.sqrt(self.price_var)
        if chosen.price <= fair_price_upper and self.wealth >= chosen.price:
            self.wealth -= chosen.price
            self.total_spent += chosen.price
            chosen.sell_to(self)
            self.purchases += 1
            self.update_price_belief(chosen.price)
            # 记录到交易网络
            self.model.network.add_transaction(self.unique_id, chosen.unique_id, chosen.price)
        # 破产检查
        if self.wealth < 1.0:
            self.alive = False
print("BuyerAgent 定义完成")
print(f"  贝叶斯先验: 公平价格 ~ N(mu=20, var=25)")
print(f"  有限理性: 70%exploit / 30%explore")

## 3. TODO 2：卖方Agent（A2A协商 + 推理成本）

**卖方Agent** 是Agent经济中的供给方（营销映射：媒介Agent卖广告流量）。

核心属性：
- `wealth`：资金（初始500）
- `base_cost`：基础成本（产品差异化，随机0.8-1.3倍）
- `price`：当前售价（动态调整）
- `supply`：供应量
- `a2a_negotiations`：A2A协商次数
- `total_reasoning_cost`：累计推理成本

**行为逻辑**：
1. A2A交易：收取价格，支付10%协议费 + 推理成本
2. 供应高则降价，供应低则涨价（动态定价）
3. 定期补货（消耗资金）
4. 资金为负则破产

**真实参数**：A2A协议费10% + 推理成本$0.0025/协商。

In [ ]:
# 卖方Agent：动态定价 + A2A协商 + 推理成本约束
class SellerAgent(mesa.Agent):
    """卖方Agent（营销映射：媒介Agent卖广告流量）。
    动态定价 + A2A协商交易 + 推理成本约束。"""
    def __init__(self, model, initial_wealth=500.0, base_cost=10.0):
        super().__init__(model)
        self.wealth = initial_wealth
        # 产品差异化：基础成本随机化（0.8-1.3倍）
        self.base_cost = base_cost * model.random.uniform(0.8, 1.3)
        # 初始定价：成本×随机加价（1.5-2.5倍）
        markup = model.random.uniform(1.5, 2.5)
        self.price = self.base_cost * markup
        self.supply = 100
        self.alive = True
        self.sales = 0
        self.revenue = 0.0
        self.a2a_negotiations = 0
        self.total_reasoning_cost = 0.0
    def sell_to(self, buyer):
        """A2A交易：收取价格，支付协议费+推理成本，动态调价。""\"
        revenue = self.price
        protocol_fee = revenue * A2A_PROTOCOL_FEE_RATE  # 10% A2A协议费
        reasoning_cost = REASONING_COST_PER_NEGOTIATION
        self.wealth += revenue - protocol_fee - reasoning_cost
        self.revenue += revenue
        self.supply -= 1
        self.sales += 1
        self.a2a_negotiations += 1
        self.total_reasoning_cost += reasoning_cost
        self._adjust_price()
    def _adjust_price(self):
        # 供应高降价，供应低涨价
        if self.supply > 60:
            self.price = max(self.base_cost, self.price * 0.97)
        elif self.supply < 30:
            self.price = self.price * 1.03
    def step(self):
        if not self.alive:
            return
        # 补货
        restock_cost = max(0, (100 - self.supply)) * self.base_cost * 0.5
        if self.wealth >= restock_cost and self.supply < 50:
            self.wealth -= restock_cost
            self.supply = 100
        if self.wealth < 0:
            self.alive = False
print("SellerAgent 定义完成")
print(f"  真实参数: A2A协议费 {A2A_PROTOCOL_FEE_RATE*100:.0f}% + 推理成本 ${REASONING_COST_PER_NEGOTIATION:.4f}/协商")

## 4. TODO 3：Agent交易网络（networkx）

**AgentTransactionNetwork** 用networkx分析Agent间A2A交易拓扑。

核心功能：
- `add_transaction(buyer_id, seller_id, amount)`：添加有向交易边
- `compute_metrics()`：计算网络密度/聚类系数/边数/节点数
- `top_sellers_by_pagerank(top_n)`：PageRank经济影响力排名

**经济意义**：网络密度反映市场交易紧密程度，PageRank识别经济hub Agent。

In [ ]:
# Agent交易网络：用networkx分析A2A交易拓扑
class AgentTransactionNetwork:
    """Agent交易网络：有向图，边=交易关系，权重=累计交易额。"""
    def __init__(self):
        self.G = nx.DiGraph()
    def add_transaction(self, buyer_id, seller_id, amount):
        """添加或更新 buyer->seller 交易边。"""
        if not self.G.has_edge(buyer_id, seller_id):
            self.G.add_edge(buyer_id, seller_id, weight=0.0, count=0)
        self.G[buyer_id][seller_id]['weight'] += amount
        self.G[buyer_id][seller_id]['count'] += 1
    def compute_metrics(self):
        """计算网络拓扑指标。"""
        n_nodes = self.G.number_of_nodes()
        if n_nodes == 0:
            return {'density': 0.0, 'avg_clustering': 0.0, 'n_edges': 0, 'n_nodes': 0}
        undirected = self.G.to_undirected()
        return {
            'density': nx.density(self.G),
            'avg_clustering': nx.average_clustering(undirected),
            'n_edges': self.G.number_of_edges(),
            'n_nodes': n_nodes,
        }
    def top_sellers_by_pagerank(self, top_n=3):
        """PageRank经济影响力排名。"""
        if self.G.number_of_nodes() == 0:
            return []
        pr = nx.pagerank(self.G)
        return sorted(pr.items(), key=lambda x: x[1], reverse=True)[:top_n]
print("AgentTransactionNetwork 定义完成")
print(f"  networkx {nx.__version__}: DiGraph + density + clustering + pagerank")

## 5. TODO 4：Agent经济模型 + DataCollector

**AgentEconomyModel** 整合买方/卖方Agent和交易网络，用DataCollector追踪涌现指标：

| 指标 | 含义 |
|------|------|
| `gini` | 基尼系数（财富不平等程度） |
| `avg_price` | 市场平均价格 |
| `price_std` | 价格标准差 |
| `n_alive_*` | 各类Agent存活数 |
| `total_a2a_trades` | 累计A2A交易量 |
| `network_density` | 交易网络密度 |
| `total_reasoning_cost` | 累计推理成本 |

**天道推演映射**：模型每一步step()就是一次沙盘推演。

In [ ]:
# Agent经济模型 + DataCollector
class AgentEconomyModel(mesa.Model):
    """Agent经济仿真模型：买方/卖方A2A协商，涌现价格/网络/财富分布。"""
    def __init__(self, n_buyers=20, n_sellers=5, seed=42):
        super().__init__(rng=seed)
        # 创建Agent
        BuyerAgent.create_agents(model=self, n=n_buyers, initial_budget=1000.0, demand=1)
        SellerAgent.create_agents(model=self, n=n_sellers, initial_wealth=500.0, base_cost=10.0)
        # 交易网络
        self.network = AgentTransactionNetwork()
        # 数据收集器
        self.datacollector = DataCollector(
            model_reporters={
                'gini': self._compute_gini,
                'avg_price': self._compute_avg_price,
                'price_std': self._compute_price_std,
                'n_alive_buyers': lambda m: sum(1 for a in m.agents if isinstance(a, BuyerAgent) and a.alive),
                'n_alive_sellers': lambda m: sum(1 for a in m.agents if isinstance(a, SellerAgent) and a.alive),
                'total_a2a_trades': lambda m: sum(a.a2a_negotiations for a in m.agents if isinstance(a, SellerAgent)),
                'network_density': lambda m: m.network.compute_metrics()['density'],
                'network_edges': lambda m: m.network.compute_metrics()['n_edges'],
                'total_reasoning_cost': lambda m: sum(a.total_reasoning_cost for a in m.agents if isinstance(a, SellerAgent)),
            },
            agent_reporters={
                'wealth': 'wealth',
                'agent_type': lambda a: type(a).__name__,
                'alive': 'alive',
            }
        )
        self.datacollector.collect(self)
    def _compute_gini(self):
        wealths = sorted([a.wealth for a in self.agents if a.wealth > 0])
        n = len(wealths)
        if n == 0 or sum(wealths) == 0:
            return 0.0
        cum = sum((i + 1) * w for i, w in enumerate(wealths))
        return (2 * cum) / (n * sum(wealths)) - (n + 1) / n
    def _compute_avg_price(self):
        prices = [a.price for a in self.agents if isinstance(a, SellerAgent) and a.alive]
        return float(np.mean(prices)) if prices else 0.0
    def _compute_price_std(self):
        prices = [a.price for a in self.agents if isinstance(a, SellerAgent) and a.alive]
        return float(np.std(prices)) if prices else 0.0
    def step(self):
        self.agents.shuffle_do('step')
        self.datacollector.collect(self)
# 验证
model = AgentEconomyModel(n_buyers=20, n_sellers=5, seed=42)
print(f"模型创建成功: {len(model.agents)} agents")
print(f"  买方: {sum(1 for a in model.agents if isinstance(a, BuyerAgent))}")
print(f"  卖方: {sum(1 for a in model.agents if isinstance(a, SellerAgent))}")
print(f"  初始基尼: {model._compute_gini():.4f}")

## 6. TODO 5：运行仿真 + 提取数据

运行Agent经济仿真20个tick，用DataCollector提取时间序列数据到pandas DataFrame。

**关键问题**：
- 基尼系数如何变化？（财富是否越来越集中？）
- 市场价格是否收敛？
- 交易网络拓扑如何演化？
- 哪类Agent最先破产？

In [ ]:
# 运行仿真 + 提取数据
model = AgentEconomyModel(n_buyers=20, n_sellers=5, seed=42)
N_STEPS = 20
for i in range(N_STEPS):
    model.step()
# 提取数据到pandas DataFrame
model_df = model.datacollector.get_model_vars_dataframe()
agent_df = model.datacollector.get_agent_vars_dataframe()
print(f"仿真规模: {N_STEPS} ticks, {len(model.agents)} agents")
print(f"\n--- 最终状态 (tick {N_STEPS}) ---")
print(f"基尼系数: {model_df['gini'].iloc[-1]:.4f}")
print(f"平均价格: ${model_df['avg_price'].iloc[-1]:.2f}")
print(f"价格标准差: ${model_df['price_std'].iloc[-1]:.2f}")
print(f"存活买方: {int(model_df['n_alive_buyers'].iloc[-1])}/20")
print(f"存活卖方: {int(model_df['n_alive_sellers'].iloc[-1])}/5")
print(f"累计A2A交易: {int(model_df['total_a2a_trades'].iloc[-1])}")
print(f"网络密度: {model_df['network_density'].iloc[-1]:.4f}")
print(f"网络边数: {int(model_df['network_edges'].iloc[-1])}")
print(f"累计推理成本: ${model_df['total_reasoning_cost'].iloc[-1]:.4f}")
print(f"\n--- 基尼系数变化 ---")
print(f"初始: {model_df['gini'].iloc[0]:.4f}")
print(f"第10步: {model_df['gini'].iloc[10]:.4f}")
print(f"最终: {model_df['gini'].iloc[-1]:.4f}")
print(f"\n--- 价格分布 (最终tick) ---")
final_prices = sorted([a.price for a in model.agents if isinstance(a, SellerAgent) and a.alive])
print(f"价格列表: {[round(p,2) for p in final_prices]}")
print(f"价格区间: ${min(final_prices):.2f} - ${max(final_prices):.2f}")
# 交易网络拓扑
metrics = model.network.compute_metrics()
print(f"\n--- 交易网络拓扑 (networkx) ---")
print(f"节点数: {metrics['n_nodes']}")
print(f"边数: {metrics['n_edges']}")
print(f"密度: {metrics['density']:.4f}")
print(f"平均聚类系数: {metrics['avg_clustering']:.4f}")
# Top卖方PageRank
top_sellers = model.network.top_sellers_by_pagerank(3)
print(f"\n--- Top 3 经济影响力 (PageRank) ---")
for node_id, pr in top_sellers:
    matching = [a for a in model.agents if a.unique_id == node_id]
    if matching:
        agent = matching[0]
        if isinstance(agent, SellerAgent):
            print(f"  Agent {node_id}: PageRank={pr:.4f}, sales={agent.sales}, revenue=${agent.revenue:.2f}")
        else:
            print(f"  Agent {node_id}: PageRank={pr:.4f}, type=Buyer, purchases={agent.purchases}")
print(f"\nmodel_df 形状: {model_df.shape}")
print(f"agent_df 形状: {agent_df.shape}")

## 7. TODO 6：Agent经济价值分析（NPV/IRR）+ 可视化

用 **numpy-financial** 计算Agent-as-Worker的经济价值，用matplotlib绘制4个子图：

1. **NPV/IRR分析**：对比Agent-as-Worker vs Human Worker的12月现金流NPV
2. **基尼系数**随时间变化
3. **市场价格分布**（均值±标准差）
4. **Agent存活数 + A2A交易网络密度**

**Agent经济价值**：Agent-as-Worker有初始部署成本但推理成本远低于人类工资。

In [ ]:
# Agent经济价值分析（NPV/IRR）+ 可视化
# Agent-as-Worker现金流量（12月）
agent_setup_cost = -2000  # 一次性Agent部署成本
agent_monthly_reasoning = -REASONING_COST_PER_NEGOTIATION * 1000  # 每月1000次A2A协商
agent_monthly_revenue = 6000  # Agent工作月收入
agent_monthly_net = agent_monthly_revenue + agent_monthly_reasoning
agent_cashflows = [agent_setup_cost] + [agent_monthly_net] * 12
# Human Worker现金流量
human_monthly_salary = -5000  # 人类月工资
human_monthly_revenue = 6000
human_monthly_net = human_monthly_revenue + human_monthly_salary
human_cashflows = [0] + [human_monthly_net] * 12
# NPV/IRR计算（numpy-financial）
discount_rate = 0.10 / 12  # 月贴现率
agent_npv = npf.npv(discount_rate, agent_cashflows)
human_npv = npf.npv(discount_rate, human_cashflows)
agent_irr = npf.irr(agent_cashflows)
print(f"\n--- Agent经济价值分析 (numpy-financial) ---")
print(f"Agent-as-Worker NPV (12月): ${agent_npv:.2f}")
print(f"Human Worker NPV (12月): ${human_npv:.2f}")
print(f"Agent-as-Worker IRR: {agent_irr*100:.2f}%")
print(f"Agent经济优势: ${agent_npv - human_npv:.2f}")
# 可视化：4子图
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# 子图1: 基尼系数
axes[0, 0].plot(model_df.index, model_df['gini'], color='#2563eb', linewidth=1.5)
axes[0, 0].set_title('基尼系数 (Agent财富不平等)', fontsize=12)
axes[0, 0].set_xlabel('Tick'); axes[0, 0].set_ylabel('Gini')
axes[0, 0].axhline(y=0.3, color='r', linestyle='--', alpha=0.5, label='0.3 警戒线')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
# 子图2: 市场价格分布
axes[0, 1].plot(model_df.index, model_df['avg_price'], color='#16a34a', linewidth=1.5, label='平均价格')
axes[0, 1].fill_between(model_df.index,
    model_df['avg_price'] - model_df['price_std'],
    model_df['avg_price'] + model_df['price_std'],
    alpha=0.2, color='#16a34a', label='+/-1 std')
axes[0, 1].set_title('市场价格分布', fontsize=12)
axes[0, 1].set_xlabel('Tick'); axes[0, 1].set_ylabel('Price ($)')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)
# 子图3: Agent存活数
axes[1, 0].plot(model_df.index, model_df['n_alive_buyers'], label='买方Agent', color='#2563eb', linewidth=1.5)
axes[1, 0].plot(model_df.index, model_df['n_alive_sellers'], label='卖方Agent', color='#dc2626', linewidth=1.5)
axes[1, 0].set_title('Agent存活数', fontsize=12)
axes[1, 0].set_xlabel('Tick'); axes[1, 0].set_ylabel('Count')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)
# 子图4: 网络密度 + A2A交易量
ax2 = axes[1, 1]
ax2.plot(model_df.index, model_df['network_density'], color='#ea580c', linewidth=1.5, label='网络密度')
ax2.set_ylabel('Density', color='#ea580c'); ax2.set_title('A2A交易网络演化', fontsize=12)
ax2.set_xlabel('Tick')
ax2b = ax2.twinx()
ax2b.plot(model_df.index, model_df['total_a2a_trades'], color='#0891b2', linewidth=1.5, label='A2A交易量')
ax2b.set_ylabel('A2A Trades', color='#0891b2')
ax2.grid(True, alpha=0.3)
plt.suptitle('Agent经济仿真涌现 (mesa + networkx + numpy-financial)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('agent_economy_e10d1.png', dpi=100, bbox_inches='tight')
plt.show()
print("图表已保存: agent_economy_e10d1.png")
# 涌现分析
print(f"\n=== 涌现分析 ===")
print(f"1. 财富不平等: 基尼系数从 {model_df['gini'].iloc[0]:.4f} 升至 {model_df['gini'].iloc[-1]:.4f}")
print(f"   -> A2A协商中的信息不对称导致财富集中")
print(f"2. 价格收敛: 平均价格 ${model_df['avg_price'].iloc[0]:.2f} -> ${model_df['avg_price'].iloc[-1]:.2f}")
print(f"   -> 贝叶斯Agent的比价促进价格发现")
print(f"3. Agent存活: 买方 {int(model_df['n_alive_buyers'].iloc[-1])}/20, 卖方 {int(model_df['n_alive_sellers'].iloc[-1])}/5")
print(f"4. 网络拓扑: 密度 {model_df['network_density'].iloc[-1]:.4f}, 边数 {int(model_df['network_edges'].iloc[-1])}")
print(f"   -> 交易网络呈现中心化趋势（少数卖方成为hub）")
print(f"5. 推理成本: 每次${REASONING_COST_PER_NEGOTIATION:.4f}, 累计${model_df['total_reasoning_cost'].iloc[-1]:.4f}")
print(f"   -> 推理成本是A2A经济可行性的核心约束")

## 8. 天道推演 × 多Agent仿真

本仿真本质是**计算化的天道推演沙盘**：

| 天道推演能力 | 仿真对应 | 涌现产出 |
|-------------|---------|---------|
| 局势感知 | 初始Agent分布与参数 | 初始基尼/价格 |
| 因果链追踪 | Agent行为因果（购买->定价->竞争） | 价格动态 |
| 沙盘模拟（3层） | 20 tick推演 | 时间序列涌现 |
| 概率评估 | 多次运行不同seed | 结果分布 |
| 最优路径推荐 | 对比不同A2A协议参数 | 策略选择 |

**核心洞察**：Agent经济仿真让天道推演从"意识中的沙盘"变为"可计算、可复现的沙盘"。

## 交付物
- [ ] 完成的 starter.ipynb（6个TODO全部填好）
- [ ] 4个子图的仿真结果可视化
- [ ] 一段300字分析：仿真涌现了什么经济现象？推理成本对Agent的影响？网络拓扑说明了什么？